# Day 3: Differential expression with nf-core/differentialabundance

**Course:** Computational Workflows for Biomedical Data (CWBD), University of Tübingen, 2026 \
**Date:** 2026-09-30 \
**Author:** Laurens Hahn

**What:** Run nf-core/differentialabundance on the provided count matrix of all 16 NAc samples, once with all samples and once without the three contaminated ones, and compare DEG counts and the Venn overlap with Mitsi et al. 2023.

**Why:** This is the step that reproduces the paper's central numbers. The day 2 QC decides which samples can be trusted.

## Choosing the pipeline

### Which pipeline would you use to compare expression between the groups?

> We would now like to understand exactly the difference between the expression in our groups of mice. \
> Which pipeline would you use for this?

nf-core/differentialabundance, release 2.0.0 like on day 1. It starts where nf-core/rnaseq stops: it takes the count matrix in `data/`, a samplesheet and a contrasts file, runs DESeq2 and gives me result tables, volcano plots, a PCA and an HTML report.

## Preparing the inputs

### Samplesheet: which matrix column is which GEO sample?

> Have a close look at the pipeline's "Usage" page on the [nf-core docs](nf-co.re). You will need to create a samplesheet (based on the column names in the provided matrix).

The `sample` column has to match the matrix columns (`Sham_oxy_1` and so on), which only give group and replicate. To carry the day 2 contamination findings over, I correlated each column with the tutors' day 2 run and matched them one to one.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

counts = pd.read_csv("data/salmon.merged.gene_counts.tsv", sep="\t", index_col=0)
samples = [c for c in counts.columns if c != "gene_name"]

# Column names are <surgery>_<treatment>_<replicate>, the group is surgery plus treatment
sheet = pd.DataFrame({"sample": samples})
sheet[["surgery", "treatment", "replicate"]] = sheet["sample"].str.split("_", expand=True)
sheet["Condition"] = sheet["surgery"] + "_" + sheet["treatment"]

# Link each column to its GEO sample via the tutors' day 2 run of all 16 samples: correlate log counts
# and solve a one to one assignment. A plain argmax sends Sham_oxy_3 to C3 twice, because the two runs put
# the rRNA reads of C2 (22 % mouse rRNA) into different genes (Rn18s-rs5 vs Yam1), which lowers its r.
day2_map = pd.read_csv("../day_02/samples_allsamples.csv", index_col=0)
day2 = pd.read_csv("../day_02/rnaseq_out_allsamples/star_salmon/salmon.merged.gene_counts.tsv", sep="\t", index_col=0)
day2 = np.log1p(day2.drop(columns="gene_name").loc[counts.index])
day3 = np.log1p(counts[samples])
cor = pd.DataFrame({s: day2.corrwith(day3[s]) for s in samples})
rows, cols = linear_sum_assignment(-cor.values)
match = pd.Series(cor.index[rows], index=cor.columns[cols])
sheet["srx"] = match.loc[samples].values
sheet["match_r"] = [round(cor.loc[match[s], s], 4) for s in samples]
sheet["geo_title"] = day2_map.loc[sheet.srx, "title"].values
sheet["run"] = day2_map.loc[sheet.srx, "run"].values
assert (day2_map.loc[sheet.srx, "group"].str.lower().values == sheet.Condition.str.lower().values).all()

# Human rRNA contamination found on day 2 (share of reads from human rDNA)
sheet["contaminated"] = np.where(sheet.geo_title.isin(["C4", "F3", "B4"]), "yes", "no")
sheet.drop(columns=["srx", "match_r"]).to_csv("samplesheet.csv", index=False)
sheet

,sample,surgery,treatment,replicate,Condition,srx,match_r,geo_title,run,contaminated
0,Sham_oxy_1,Sham,oxy,1,Sham_oxy,SRX19144492,0.9991,C4,SRR23195506,yes
1,Sham_oxy_2,Sham,oxy,2,Sham_oxy,SRX19144488,0.9994,C3,SRR23195511,no
2,Sham_oxy_3,Sham,oxy,3,Sham_oxy,SRX19144484,0.9855,C2,SRR23195514,no
3,Sham_oxy_4,Sham,oxy,4,Sham_oxy,SRX19144480,0.9992,C1,SRR23195519,no
4,Sham_Sal_1,Sham,Sal,1,Sham_Sal,SRX19144491,0.9992,B4,SRR23195507,yes
5,Sham_Sal_2,Sham,Sal,2,Sham_Sal,SRX19144487,0.9994,B3,SRR23195512,no
6,Sham_Sal_3,Sham,Sal,3,Sham_Sal,SRX19144483,0.9994,B2,SRR23195515,no
7,Sham_Sal_4,Sham,Sal,4,Sham_Sal,SRX19144479,0.9993,B1,SRR23195520,no
8,SNI_oxy_1,SNI,oxy,1,SNI_oxy,SRX19144494,0.9993,H4,SRR23195508,no
9,SNI_oxy_2,SNI,oxy,2,SNI_oxy,SRX19144490,0.9993,H3,SRR23195509,no


### Gene lengths, clean set and contrasts

In [2]:
# Gene lengths for DESeq2 (the docs advise raw Salmon counts only together with lengths). data/ has no
# lengths matrix, rnaseq-out-small has one but with Nac_* column names. Match its columns by counts.
small = pd.read_csv("data/rnaseq-out-small/star_salmon/salmon.merged.gene_counts.tsv", sep="\t", index_col=0).drop(columns="gene_name")
lengths = pd.read_csv("data/rnaseq-out-small/star_salmon/salmon.merged.gene_lengths.tsv", sep="\t", index_col=0)
nac_to_sample = {counts[samples].sub(small[n], axis=0).abs().max().idxmin(): n for n in small.columns}
assert len(nac_to_sample) == 16
lengths = lengths.rename(columns={n: s for s, n in nac_to_sample.items()})[["gene_name"] + samples]
lengths.to_csv("salmon.merged.gene_lengths.tsv", sep="\t")
print(nac_to_sample)

# Clean set without the three contaminated samples, for the second run
keep = sheet.loc[sheet.contaminated == "no", "sample"].tolist()
sheet.loc[sheet.contaminated == "no"].drop(columns=["srx", "match_r"]).to_csv("samplesheet_clean.csv", index=False)
counts[["gene_name"] + keep].to_csv("salmon.merged.gene_counts_clean.tsv", sep="\t")
lengths[["gene_name"] + keep].to_csv("salmon.merged.gene_lengths_clean.tsv", sep="\t")

# Contrasts: the two provided (oxycodone effect within SNI and within Sham) plus the paper's three vs Sham_Sal
contrasts = pd.DataFrame([
    ["oxy_vs_sal_in_SNI", "Condition", "SNI_Sal", "SNI_oxy"],
    ["oxy_vs_sal_in_Sham", "Condition", "Sham_Sal", "Sham_oxy"],
    ["SNI_Sal_vs_Sham_Sal", "Condition", "Sham_Sal", "SNI_Sal"],
    ["Sham_oxy_vs_Sham_Sal", "Condition", "Sham_Sal", "Sham_oxy"],
    ["SNI_oxy_vs_Sham_Sal", "Condition", "Sham_Sal", "SNI_oxy"],
], columns=["id", "variable", "reference", "target"])
contrasts.to_csv("contrasts_paper.csv", index=False)
contrasts

{'Sham_oxy_1': 'Nac_oxy_1', 'SNI_oxy_1': 'Nac_oxy_2', 'SNI_oxy_2': 'Nac_oxy_3', 'Sham_oxy_2': 'Nac_oxy_4', 'Sham_oxy_3': 'Nac_oxy_5', 'SNI_oxy_3': 'Nac_oxy_6', 'SNI_oxy_4': 'Nac_oxy_7', 'Sham_oxy_4': 'Nac_oxy_8', 'SNI_Sal_1': 'Nac_Sal_1', 'Sham_Sal_1': 'Nac_Sal_2', 'SNI_Sal_2': 'Nac_Sal_3', 'Sham_Sal_2': 'Nac_Sal_4', 'SNI_Sal_3': 'Nac_Sal_5', 'Sham_Sal_3': 'Nac_Sal_6', 'SNI_Sal_4': 'Nac_Sal_7', 'Sham_Sal_4': 'Nac_Sal_8'}


,id,variable,reference,target
0,oxy_vs_sal_in_SNI,Condition,SNI_Sal,SNI_oxy
1,oxy_vs_sal_in_Sham,Condition,Sham_Sal,Sham_oxy
2,SNI_Sal_vs_Sham_Sal,Condition,Sham_Sal,SNI_Sal
3,Sham_oxy_vs_Sham_Sal,Condition,Sham_Sal,Sham_oxy
4,SNI_oxy_vs_Sham_Sal,Condition,Sham_Sal,SNI_oxy


### Reference annotation

In [3]:
# GTF of the rnaseq run that produced the matrix (pipeline_info/params: genome GRCm38, iGenomes Ensembl)
!mkdir -p reference
![ -f reference/genes_GRCm38_igenomes.gtf ] || curl -s -o reference/genes_GRCm38_igenomes.gtf https://ngi-igenomes.s3.amazonaws.com/igenomes/Mus_musculus/Ensembl/GRCm38/Annotation/Genes/genes.gtf
!ls -lh reference

total 852M
-rw-r--r-- 1 laurens laurens 852M Sep 30 12:51 genes_GRCm38_igenomes.gtf


## Running the pipeline

### Which command did you use?

> Please paste here the command you used. You may need to inspect the provided expression matrix more closely and create additional files, like a samplesheet (based on the column names) or a contrast file (there happens to also be one in `data/` that you can use).

In [ ]:
# Already run (about 5 min per run); -resume makes a rerun cheap
%%bash
# Run 1: all 16 samples. Run 2: without the three human rRNA samples (C4, F3, B4).
# Day 1's test run took about 4 min on the server, this matrix is bigger.
# Typed overrides go in a params file: in 2.0.0 command line values reach the schema check as strings
# ("Validation failed for paramsheet row: rnaseq", e.g. seed (42) is [string] but should be [integer]).
cat > params_paper.yaml <<'YAML'
contrasts: contrasts_paper.csv
gtf: reference/genes_GRCm38_igenomes.gtf
differential_max_pval: 0.05          # paper: nominal p < 0.05
differential_max_qval: 1.0
differential_min_fold_change: 1.41421  # paper: |log2FC| > 0.5, as linear fold change 2^0.5
deseq2_shrink_lfc: false
seed: 42
exploratory_main_variable: Condition
YAML

COMMON="-r 2.0.0 -profile rnaseq,docker -c ../day_02/server.config -params-file params_paper.yaml -resume"

nextflow run nf-core/differentialabundance $COMMON \
  --input samplesheet.csv \
  --matrix data/salmon.merged.gene_counts.tsv --feature_length_matrix salmon.merged.gene_lengths.tsv \
  --study_name nac_all16 --outdir results/all16 > run_all16.log 2>&1
echo "all16 exit $?"; tail -15 run_all16.log

nextflow run nf-core/differentialabundance $COMMON \
  --input samplesheet_clean.csv \
  --matrix salmon.merged.gene_counts_clean.tsv --feature_length_matrix salmon.merged.gene_lengths_clean.tsv \
  --study_name nac_clean13 --outdir results/clean13 > run_clean13.log 2>&1
echo "clean13 exit $?"; tail -15 run_clean13.log

all16 exit 0

  report:
    - [report_html, {id: nac_all16, paramset_name: rnaseq, params: {contrasts: contrasts_paper.csv, gtf: reference/genes_GRCm38_igenomes.gtf, differential_max_pval: 0.05, differential_max_qval: 1.0, differential_min_fold_change: 1.41421, deseq2_shrink_lfc: false, seed: 42, exploratory_main_variable: Condition, input: samplesheet.csv, matrix: data/salmon.merged.gene_counts.tsv, feature_length_matrix: salmon.merged.gene_lengths.tsv, study_name: nac_all16, outdir: results/all16, study_type: rnaseq, study_abundance_type: counts, querygse: null, control_features: null, sizefactors_from_controls: false, round_digits: -1, paramsheet: null, observations_type: sample, observations_id_col: sample, observations_name_col: sample, features: null, features_type: gene, features_id_col: gene_id, features_name_col: gene_name, features_metadata_cols: 'gene_id,gene_name,gene_biotype', features_gtf_feature_type: transcript, features_gtf_table_first_field: gene_id, affy_cel_files_ar

### Which parameters did you set, and why?

> Explain all the parameters you set and why you set them in this way. If you used or created additional files as input, explain what they are used for.

I ran the pipeline twice: on all 16 samples (`all16`) and without the three contaminated ones (`clean13`, see the exclusion below).

**Input files**
- `samplesheet.csv`: one row per matrix column. `Condition` (Sham_oxy, Sham_Sal, SNI_oxy, SNI_Sal) is the variable the contrasts use. `geo_title`, `run` and `contaminated` trace each sample back to day 2.
- `contrasts_paper.csv`: the two contrasts from `data/contrasts.csv` (oxy vs sal within SNI and within Sham) plus the paper's three comparisons against Sham_Sal. `reference` is the denominator, so a positive log2FC means higher in `target`.
- `--matrix`: the raw Salmon gene counts from `data/`.
- `--feature_length_matrix`: gene lengths per sample. The docs say raw Salmon counts should only be used together with lengths, which DESeq2 takes as an offset. `data/` has none, so I used the one from `data/rnaseq-out-small` and renamed its `Nac_*` columns.
- `--gtf`: GRCm38 from iGenomes, the annotation the rnaseq run behind the matrix used (its `params_*.json`). It adds gene names and biotypes.

**Parameters**
- `-r 2.0.0`: pinned release.
- `-profile rnaseq,docker`: `rnaseq` sets DESeq2 and the column names for a count matrix, `docker` runs each step in its container.
- `-c ../day_02/server.config`: keeps the run within the server (12 CPUs, 14 GB).
- `differential_max_pval 0.05`, `differential_max_qval 1`, `differential_min_fold_change 1.41421`: the paper's cutoff (nominal p < 0.05 and |log2FC| > 0.5, and 2^0.5 = 1.414), so my counts compare to theirs. The defaults would be q < 0.05 and fold change 2. I count padj < 0.05 separately below.
- `deseq2_shrink_lfc false`: the paper used plain fold changes, shrinking would push genes below the 0.5 cutoff.
- `seed 42`: reproducible random steps.
- `exploratory_main_variable: Condition`: with the default `auto_pca` the 2.0.0 report crashes ("Insufficient values in manual scale. 4 needed but only 2 provided").
- `-resume`: reuses finished steps after a fix.

Two workarounds for 2.0.0: typed values given on the command line reach the schema check as strings and fail, so thresholds, seed and shrinkage are in `params_paper.yaml` (`-params-file`). And `exclude_samples_col` in the contrasts file is never passed to DESeq2, so the clean set is a second run with its own samplesheet and matrices.

**Model:** `~ Condition`, one mean per group. The paper's `~ SNI + oxy + SNI:oxy` is the same model with other coefficients, so every group vs Sham_Sal comparison is identical.

### What were the outputs of the pipeline?

> What were the outputs of the pipeline?

Each run folder (`results/all16`, `results/clean13`) contains:
- `tables/differential`: per contrast the full DESeq2 table (log2FoldChange, pvalue, padj), a filtered one with only the genes passing my thresholds, and an annotated one with gene names.
- `tables/processed_abundance`: normalised and VST counts.
- `plots/exploratory` (PCA, clustering, density, outlier scores), `plots/differential` (a volcano plot per contrast), `plots/qc` (dispersion).
- `report`: the HTML report, the main thing to open. `shinyngs_app` has the same data as an interactive app.
- `other/deseq2`: the DESeq2 objects. `pipeline_info`: run reports and software versions.

In [4]:
# What did the pipeline write? Folder tree two levels deep plus the differential tables
!find results/all16 -maxdepth 3 -type d | sort
!find results/all16 -name "*.deseq2.results.tsv" | sort
!find results/all16 -name "*.html" | sort

results/all16
results/all16/other
results/all16/other/deseq2
results/all16/other/deseq2/rnaseq
results/all16/pipeline_info
results/all16/plots
results/all16/plots/differential
results/all16/plots/differential/rnaseq
results/all16/plots/exploratory
results/all16/plots/exploratory/rnaseq
results/all16/plots/qc
results/all16/plots/qc/rnaseq
results/all16/report
results/all16/report/rnaseq
results/all16/shinyngs_app
results/all16/shinyngs_app/rnaseq
results/all16/shinyngs_app/rnaseq/nac_all16
results/all16/tables
results/all16/tables/annotation
results/all16/tables/annotation/rnaseq
results/all16/tables/differential
results/all16/tables/differential/rnaseq
results/all16/tables/processed_abundance
results/all16/tables/processed_abundance/rnaseq


results/all16/tables/differential/rnaseq/SNI_Sal_vs_Sham_Sal_nac_all16.deseq2.results.tsv
results/all16/tables/differential/rnaseq/SNI_oxy_vs_Sham_Sal_nac_all16.deseq2.results.tsv
results/all16/tables/differential/rnaseq/Sham_oxy_vs_Sham_Sal_nac_all16.deseq2.results.tsv
results/all16/tables/differential/rnaseq/oxy_vs_sal_in_SNI_nac_all16.deseq2.results.tsv
results/all16/tables/differential/rnaseq/oxy_vs_sal_in_Sham_nac_all16.deseq2.results.tsv


results/all16/pipeline_info/execution_report_2026-09-30_13-05-13.html
results/all16/pipeline_info/execution_report_2026-09-30_13-08-41.html
results/all16/pipeline_info/execution_report_2026-09-30_13-20-02.html
results/all16/pipeline_info/execution_timeline_2026-09-30_13-05-13.html
results/all16/pipeline_info/execution_timeline_2026-09-30_13-08-41.html
results/all16/pipeline_info/execution_timeline_2026-09-30_13-20-02.html
results/all16/pipeline_info/pipeline_dag_2026-09-30_13-05-13.html
results/all16/pipeline_info/pipeline_dag_2026-09-30_13-08-41.html
results/all16/pipeline_info/pipeline_dag_2026-09-30_13-20-02.html
results/all16/report/rnaseq/nac_all16_differentialabundance_report.html


## Sample quality

### Would you exclude any samples?

> Would you exclude any samples? If yes, which and why?

In [5]:
import shutil

# The pipeline's PCA of the all16 run, kept next to the notebook
shutil.copy("results/all16/plots/exploratory/rnaseq/Condition/png/pca2d.png", "figures/pca2d_all16.png")

# Library size (assigned reads) per sample next to the day 2 contamination findings
human_rRNA = {"C4": 58.4, "F3": 29.8, "B4": 24.1, "F2": 16.8, "B1": 10.6, "C1": 9.9}   # % of R1 reads, day 2
mouse_rRNA = {"F1": 31.1, "C2": 22.1}                                                 # % of R1 reads, day 2
qc = sheet.set_index("sample")[["Condition", "geo_title", "run"]].copy()
qc["assigned_M"] = (counts[samples].sum() / 1e6).round(2)
qc["human_rRNA_%"] = qc.geo_title.map(human_rRNA)
qc["mouse_rRNA_%"] = qc.geo_title.map(mouse_rRNA)
qc.sort_values("assigned_M")

,Condition,geo_title,run,assigned_M,human_rRNA_%,mouse_rRNA_%
sample,,,,,,
Sham_oxy_1,Sham_oxy,C4,SRR23195506,2.77,58.4,NaN
SNI_Sal_2,SNI_Sal,F3,SRR23195510,6.83,29.8,NaN
Sham_Sal_1,Sham_Sal,B4,SRR23195507,7.01,24.1,NaN
Sham_oxy_3,Sham_oxy,C2,SRR23195514,7.08,NaN,22.1
SNI_Sal_3,SNI_Sal,F2,SRR23195513,7.71,16.8,NaN
Sham_oxy_2,Sham_oxy,C3,SRR23195511,7.97,NaN,NaN
SNI_oxy_4,SNI_oxy,H1,SRR23195517,8.21,NaN,NaN
Sham_oxy_4,Sham_oxy,C1,SRR23195519,10.13,9.9,NaN
SNI_Sal_4,SNI_Sal,F1,SRR23195518,10.40,NaN,31.1


Yes, three: **Sham_oxy_1 (C4, SRR23195506), SNI_Sal_2 (F3, SRR23195510) and Sham_Sal_1 (B4, SRR23195507)**. On day 2 all three carried human rRNA (58.4, 29.8 and 24.1 % of R1 reads). Foreign reads can land on mouse genes, so their profiles are shifted for a technical reason, not a biological one.

- C4 also has by far the smallest library (2.8 M assigned reads, the others 7 to 16 M) and sits alone at the top of PC2 (17 %).
- PC1 (22 %) is driven by F3 and F1 alone, and the four groups do not separate. So the largest variation in the data is technical.
- B4 does not stand out on the PCA and is the weakest case, but 24 % human reads is too much to keep.
- F1 (SNI_Sal_4) is a PCA outlier too, but its extra reads are mouse rRNA (31 %), which mostly land on rRNA genes. Dropping it would leave only 2 SNI_Sal samples, so I keep it flagged. The same goes for C2 (Sham_oxy_3, 22 % mouse rRNA), which is no outlier.
- With 4 samples per group every removal costs power, which is why I ran both sets.

![PCA all16](figures/pca2d_all16.png)

*PCA of all 16 samples from the pipeline report (VST, top 500 variable genes). F3 and F1 pull PC1, C4 sits alone at the top of PC2, and the four groups do not separate.*

## Differential expression

### How many genes are differentially expressed per contrast, and does this confirm the paper?

> How many genes were differentially expressed in each contrast? Does this confirm what the paper mentions?

In [3]:
import glob

def load_results(run, contrast):
    path = glob.glob(f"results/{run}/**/{contrast}*.deseq2.results.tsv", recursive=True)
    assert len(path) == 1, path
    return pd.read_csv(path[0], sep="\t", index_col=0)

def paper_degs(res):
    # Paper cutoff: nominal p < 0.05 and |log2FC| > 0.5
    return set(res.index[(res.pvalue < 0.05) & (res.log2FoldChange.abs() > 0.5)])

paper = {"SNI_Sal_vs_Sham_Sal": 1457, "Sham_oxy_vs_Sham_Sal": 2609, "SNI_oxy_vs_Sham_Sal": 1012}
rows = []
for contrast in contrasts.id:
    row = {"contrast": contrast, "paper": paper.get(contrast)}
    for run in ["all16", "clean13"]:
        res = load_results(run, contrast)
        row[f"{run}_p<0.05"] = len(paper_degs(res))
        row[f"{run}_padj<0.05"] = int(((res.padj < 0.05) & (res.log2FoldChange.abs() > 0.5)).sum())
    # Direction in the clean run: up = higher in the target group
    fc = res.loc[list(paper_degs(res)), "log2FoldChange"]
    row["clean13_up"], row["clean13_down"] = int((fc > 0).sum()), int((fc < 0).sum())
    rows.append(row)
deg_table = pd.DataFrame(rows).set_index("contrast")
deg_table

,paper,all16_p<0.05,all16_padj<0.05,clean13_p<0.05,clean13_padj<0.05,clean13_up,clean13_down
contrast,,,,,,,
oxy_vs_sal_in_SNI,NaN,969,19,826,3,257,569
oxy_vs_sal_in_Sham,NaN,1280,46,1531,364,1174,357
SNI_Sal_vs_Sham_Sal,1457.0,1538,149,2042,426,1575,467
Sham_oxy_vs_Sham_Sal,2609.0,1280,46,1531,364,1174,357
SNI_oxy_vs_Sham_Sal,1012.0,669,3,1198,191,951,247


At the paper's cutoff (nominal p < 0.05, |log2FC| > 0.5), padj < 0.05 with the same fold change in brackets, and the direction for the clean run (up means higher in the target group):

| Contrast | All 16 | Without C4, F3, B4 | Up / down (without C4, F3, B4) | Paper (NAc) |
|---|---|---|---|---|
| SNI-Sal vs Sham-Sal | 1538 (149) | 2042 (426) | 1575 / 467 | 1457 |
| Sham-Oxy vs Sham-Sal | 1280 (46) | 1531 (364) | 1174 / 357 | 2609 |
| SNI-Oxy vs Sham-Sal | 669 (3) | 1198 (191) | 951 / 247 | 1012 |
| Oxy vs Sal within SNI | 969 (19) | 826 (3) | 257 / 569 | |
| Oxy vs Sal within Sham | 1280 (46) | 1531 (364) | 1174 / 357 | |

Oxy vs Sal within Sham is the same comparison as Sham-Oxy vs Sham-Sal, hence the identical numbers. Most contrasts have far more up than down regulated genes. Only oxycodone within SNI goes the other way (569 down, 257 up), so withdrawal in pain mostly switches genes off relative to pain alone.

**Does this confirm the paper?** Only partly. Without the three contaminated samples every count goes up and lands in the paper's range (826 to 2042 vs 1012 to 2609), and SNI-Oxy comes close (1198 vs 1012). But the ranking differs: in the paper withdrawal without pain (Sham-Oxy, 2609) changes the most genes, here pain alone does (SNI-Sal, 2042), and Sham-Oxy stays well below the paper (1531).

Why it cannot match exactly:
- Different processing: HISAT2 + HTSeq on mm10 (Ensembl 90) in the paper, STAR + Salmon on GRCm38 here. The pipeline also filters leniently (1 count in 1 sample is enough), so about 26,500 genes are tested.
- The paper does not say whether it removed samples.
- The cutoff is loose. At p < 0.05 about 1,300 of 26,000 genes pass by chance before the fold change filter. With padj < 0.05 only 191 to 426 remain, and with all 16 samples almost none (3 for SNI-Oxy). Many of the paper's DEGs are therefore likely false positives, and what holds is the direction: pain and withdrawal both change NAc expression, and the changes overlap.

### Can we reproduce the Venn diagram of Figure 3?

> Please reproduce the Venn Diagram from Figure 3, not taking into account the brain regions but just the contrasts mentionned.

I used the three contrasts against Sham_Sal at the paper's cutoff. Brain regions do not matter here, since all 16 samples are NAc.

In [4]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Regions as (in set 1, in set 2, in set 3) with the (x, y) of their label
REGIONS = {
    (1, 0, 0): (-1.0, 0.55), (0, 1, 0): (1.0, 0.55), (0, 0, 1): (0, -1.2),
    (1, 1, 0): (0, 0.8), (1, 0, 1): (-0.62, -0.4), (0, 1, 1): (0.62, -0.4), (1, 1, 1): (0, 0),
}

def region_counts(sets):
    counts = {}
    for flags in REGIONS:
        inside = [s for s, f in zip(sets, flags) if f]
        outside = [s for s, f in zip(sets, flags) if not f]
        counts[flags] = len(set.intersection(*inside) - set().union(*outside))
    return counts

def venn3(counts, labels, title, ax):
    for flags, (x, y) in REGIONS.items():
        ax.text(x, y, counts[flags], ha="center", va="center", fontsize=12)
    for (x, y), lab, col, (tx, ty) in zip([(-0.55, 0.3), (0.55, 0.3), (0, -0.6)], labels,
                                          ["tab:blue", "tab:orange", "tab:green"],
                                          [(-1.2, 1.75), (1.2, 1.75), (0, -2.15)]):
        ax.add_patch(Circle((x, y), 1.1, alpha=0.3, color=col))
        ax.text(tx, ty, lab, ha="center", fontsize=10)
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.4, 2.1); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title)

venn_contrasts = ["Sham_oxy_vs_Sham_Sal", "SNI_Sal_vs_Sham_Sal", "SNI_oxy_vs_Sham_Sal"]
labels = ["Sham-Oxy vs Sham-Sal", "SNI-Sal vs Sham-Sal", "SNI-Oxy vs Sham-Sal"]

# Paper, Fig. 3b (NAc), region counts read off the figure in the same set order
paper_venn = {(1, 0, 0): 1538, (0, 1, 0): 543, (0, 0, 1): 261,
              (1, 1, 0): 407, (1, 0, 1): 244, (0, 1, 1): 87, (1, 1, 1): 420}
assert [sum(n for f, n in paper_venn.items() if f[k]) for k in range(3)] == [2609, 1457, 1012]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
venn3(paper_venn, labels, "Paper, Mitsi et al. 2023 Fig. 3b (NAc)", axes[0])
for ax, run, name in zip(axes[1:], ["all16", "clean13"], ["All 16 samples", "Clean 13 (without C4, F3, B4)"]):
    sets = [paper_degs(load_results(run, c)) for c in venn_contrasts]
    venn3(region_counts(sets), labels, name, ax)
    print(run, "shared by all three:", len(set.intersection(*sets)))
plt.tight_layout()
!mkdir -p figures
plt.savefig("figures/venn_fig3_nac.png", dpi=150, bbox_inches="tight")
plt.close()

all16 shared by all three: 187
clean13 shared by all three: 577


![Venn](figures/venn_fig3_nac.png)

*DEGs at the paper's cutoff shared between the three contrasts against Sham_Sal: as published (Fig. 3b, NAc), for all 16 samples, and without C4, F3, B4.*

Partly. The shared core reproduces: without C4, F3 and B4, 577 genes are DEGs in all three contrasts, against 420 in the paper (with all 16 samples only 187). The rest of the diagram looks different. In the paper most unique genes belong to Sham-Oxy (1538 of 2609), so withdrawal without pain has the largest signature of its own. In the clean run it is SNI-Sal (1029 of 2042), and Sham-Oxy has only 492 unique genes. So the common core that changes with pain and with withdrawal holds up, but which condition drives the most change does not.

## Biological context

### What are the NAc, mPFC and VTA?

> The paper mentions differentially expressed genes in three brain regions : the NAc, mPFC and VTA. Briefly explain what these 3 regions are.

- **NAc (nucleus accumbens):** part of the ventral striatum and the hub of the reward circuit. Mostly GABAergic medium spiny neurons (D1 and D2 type) that combine dopamine input from the VTA with glutamate input from the mPFC. Central to drug reward, motivation and withdrawal. The 16 samples here show the striatal markers Ppp1r1b, Penk, Drd1 and Drd2 (day 2), which fits.
- **mPFC (medial prefrontal cortex):** frontal cortex for decision making and emotion control. It projects to the NAc and VTA and is involved in pain and in the loss of control in addiction.
- **VTA (ventral tegmental area):** midbrain nucleus with the dopamine neurons that project to the NAc (mesolimbic) and to the mPFC (mesocortical). Opioids activate them by inhibiting the local GABA interneurons.

Together they form the mesocorticolimbic reward system. The paper punched all three 21 days into withdrawal.

### Can we tell from the paper which genes are differentially expressed in each region?

> Is there anyway from the paper and the material and methods for us to know which genes are included in these regions?

No, not from the paper and its Methods alone. They name the tools and give only numbers per region (NAc 1,012 / 1,457 / 2,609, mPFC 1,116 / 1,052 / 1,449, VTA 533 / 425 / 584) and the overlaps (420 shared DEGs in NAc, 192 in mPFC, 19 in VTA), but no gene lists. Those would be in the supplementary tables, which I could not open (paywall, still to check).

Rebuilding them ourselves does not work either: GEO GSE223541 only holds the 16 NAc samples used here. The mPFC and VTA reads were never deposited.

### Is a list of DEGs enough for the biologists, and what does the paper do beyond it?

> Once you have your list of differentially expressed genes, do you think just communicating those to the biologists would be sufficient? What does the publication state?

No. A DEG list on its own is long, contains many false positives at a nominal cutoff and says nothing about what the changes mean. The paper goes several steps further:
- **Pathways and upstream regulators** with IPA and GO, which point to HDAC1 as a top regulator in NAc and mPFC.
- **RRHO** (rank rank hypergeometric overlap), which compares whole ranked gene lists without any cutoff.
- **Cell type deconvolution** with CIBERSORTx against a single cell NAc reference, to separate changes in cell composition (neurons, microglia) from changes within cells.
- **Validation** of Hdac1 by RNAscope and immunofluorescence in neurons and microglia.
- **A functional test:** the HDAC1/2 inhibitor RBC1HI reduces withdrawal behaviour, more strongly in mice with nerve injury.

So the DEGs are the starting point for the HDAC1/2 story, not the result.